# Publaynet dataset
This dataset is used to recognize the tables 

In [ ]:
#  The dataset is being prepared
from datasets import load_dataset
import pprint
from datasets import Dataset
import matplotlib.pyplot as plt #This is used to build the actual white canvas for the image to put
import matplotlib.patches as patches 

# Streamlining the pyblaynet dataset from hugging face datasets library
ds = load_dataset(
    "jordanparker6/publaynet",
    split="train",
    streaming=True
)

# inspecting few samples
for i, sample in enumerate(ds):

    print("KEYS: ", sample.keys())
    
    # depth usually means the number of boxes I want to go inside
    pprint.pprint(sample,depth = 4)

    if i == 2:
        break

# Taking 1001 raw data from the dataset and storing in the form of list
small_samples = []
for i, sample in enumerate(ds):
    small_samples.append(sample)

    if i >= 1000:
        break

# converting the python list to a hugging face dataset
small_ds = Dataset.from_list(small_samples)

# Visualizing the 3rd image 
sample = small_ds[3] 

image = sample["image"]
annotations = sample["annotations"]

# fig = figure and ax = axes 
# fig is the window that holds everything if I want to change the color or save the image then I can use fig
# ax is the plot the graphing area containing the x-axis, and the y-axis, the gridlines and the data.
fig, ax = plt.subplots(figsize=(10,10))

# imshow() method is desinged to render 2d pictures 
ax.imshow(image)

for annotation in annotations:
    x, y, w, h = annotation["bbox"]
    rect = patches.Rectangle(
        (x, y),
        w, 
        h,
        linewidth=2,
        # edgecolor defines the border of the rectangle
        edgecolor='red',
        # facecolor defines the inside of the rectangle
        facecolor='none' 
    )
    ax.add_patch(rect)

plt.show()

In [ ]:
#  Testing the preprocessed pipeline
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as F
from datasets import load_dataset

# The hugging face dataSet are already parsed 
# build the pytorch dataloader
class publayNetDataset(Dataset):
    def __init__(self, small_ds, transform):
        self.small_ds = small_ds
        self.transform = transform
    
    def __len__(self):
        return len(self.small_ds)

    # Bundle the python box and label tensors into python dictionary
    # Finally return the processed image tensor and the target dictionary 

     # Below function load a image using ovencv or PIL library
    def __getitem__(self, index):
        sample = self.small_ds[index]
        image = sample['image']
        
        # annotations
        annotations = sample['annotations']

        boxes = []
        labels = []

        
        # finds labels and bbox with that particular image
        for ann in annotations:
            bbox = ann['bbox']
            category = ann['category_id']

            x_min, y_min, w, h = bbox
            x_max = x_min + w
            y_max = y_min + h

            # stored each coordinate of boxes in python list
            if w>0 and h>0:
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(category)

        # Ensure the box has width and height greater than 0 
        # Convert the python list into tensors, boxes into float tensors and labels into integer tensors
        if len(boxes) > 0:
            boxes_tensor = torch.as_tensor(boxes, dtype = torch.float32)
        else:
            boxes_tensor = torch.empty((0, 4), dtype = torch.float32)
        label_tensor = torch.as_tensor(labels, dtype = torch.int64)

        # Wrapping the tensor objects in the dictionaries
        target = {
            "boxes": boxes_tensor,
            "labels": label_tensor
        }
        
        # Pass the loaded image through transformation pipeline for resizing and normalization
        if hasattr(self, 'transform') and self.transform is not None:
            image, target = self.transform(image, target)
        else:
            image = F.to_tensor(image)
                
        return image, target


# Instantiate your custom TableDataset class
my_table_dataset = publayNetDataset(small_ds, transform=None)

#  Print the length to check the __len__ function
print(f"Total samples in dataset: {len(my_table_dataset)}")

#  Grab the very first preprocessed item using __getitem__
image_tensor, target_dict = my_table_dataset[3]

# 5. Print out the results to verify everything worked!
print("\n--- Preprocessing Output Verification ---")
print(f"Image Tensor Shape: {image_tensor.shape}")
print(f"Boxes Tensor Shape: {target_dict['boxes'].shape}")
print(f"Labels Tensor Shape: {target_dict['labels'].shape}")
print(f"Target Labels Content: {target_dict['labels']}")
    

# MJSynth DataSet

This dataset will be used to identify the text below the preprocessing of the dataset will be done

In [ ]:
#  The dataset is being prepared here

from datasets import load_dataset
import pprint
from datasets import Dataset
import matplotlib.pyplot as plt


ds2 = load_dataset(
    "priyank-m/MJSynth_text_recognition",
     split= "train",
     streaming = True
)

# Inspecting the few samples 
for i, sample in enumerate(ds2):
    print("keys", sample.keys())

    pprint.pprint(sample, depth=2)

    if i == 2:
        break


# Storing 1000 dataset in form of python list
small_sample = []
for i, sample in enumerate(ds2):
    small_sample.append(sample)

    if i > 999:
        break


# Convert the python list into hugginf face dataset
small_ds2 = Dataset.from_list(small_sample)

# Visualizing one image
sample = small_ds2[230]
image = sample["image"]

# Plot the image in the 10*10 plot
# fig can be used to change the color or save the image 
# ax can be used to make changes in axes
fig, ax = plt.subplots(figsize = (10, 4))

# to plot the image 
ax.imshow(image)
ax.axis('off')
plt.show()


In [ ]:
# The preprocessing of the dataset is done here

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.transforms.functional as F
import torchvision.transforms as T
from datasets import load_dataset

# create the custom collection function
class MJSynthDataset(Dataset):
    def __init__(self, small_ds2, char_to_idx_dict, transform=None):
        self.small_ds2 = small_ds2
        self.char_to_idx_dict = char_to_idx_dict
        self.transform = transform

    def __len__(self):
        return len(self.small_ds2)

    def __getitem__(self, index):

        # Grapping a sample from the dataset
        sample = self.small_ds2[index]
        image = sample["image"]
        label = sample["label"]

        # transform the image converting in grayscale, resize to height=32, convert to tensor
        if self.transform is not None:
            image_tensor = self.transform(image)
        else:
            image_tensor = F.to_tensor(image)

        # convert characters into integers_id
        integer_ids = []
        for char in label:
            index = self.char_to_idx_dict.get(char, 0)
            integer_ids.append(index)

        label_tensor = torch.tensor(integer_ids, dtype=torch.int64)

        # Return the processed pieces
        return image_tensor, label_tensor

# create collate fn different lebel tensors sizes and convert them to the same length
# pytorch will hand the batch automatically whatever the getitem returns
def crnn_collate_fn(batch):
    images = []
    labels = []
    labels_length = []

    # unpack the list of tuples 
    for image_tensor, label_tensor in batch:
        images.append(image_tensor)
        labels.append(label_tensor)
        labels_length.append(len(label_tensor))

    images_tensor = torch.stack(images)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)
    labels_lengths_tensor = torch.tensor(labels_length, dtype=torch.int64)
    return images_tensor, labels_padded, labels_lengths_tensor


# scan the dataset to pull out the unique characters
unique_chars = sorted(list(set("".join(small_ds2["label"]))))

# Build the dictionary with index 0 reserved for the CTC token
my_dictionary = {char: idx for idx, char in enumerate(unique_chars, start=1)}
my_dictionary["[blank]"] = 0

# passing the objects through transformers and converting into tensors
my_transform_pipeline = T.Compose([
    T.Resize((32, 128)), 
    T.ToTensor()
])

# initialize the dataset class 
my_crnn_dataset = MJSynthDataset(
    small_ds2=small_ds2,
    char_to_idx_dict=my_dictionary,
    transform=my_transform_pipeline
)

# plug it into the dataLoader
my_dataloader = DataLoader(
    dataset=my_crnn_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=crnn_collate_fn
)

In [ ]:
#  Test the preprocessed dataloaders
import matplotlib.pyplot as plt

images, labels, labels_length = next(iter(my_dataloader))

print("Batch Shapes")

# printing the batch size, channels, height and width
print(f"image tensor: {images.shape}")

# printing the batch size and max_sequnce_length
print(f"label tensor: {labels.shape}")

# printing the batch size
print(f"lengths Tensor: {labels_length.shape}")

# 3. Create a reverse dictionary (Index to Character)
# This lets us translate the numbers back into human text
idx_to_char = {idx: char for char, idx in my_dictionary.items()}

# 4. Isolate the very first item in the batch
first_image = images[3]
first_label = labels[3]
first_length = labels_length[3]

# 5. Decode the padded label back into text
decoded_text = ""
for idx in first_label.numpy():
    # We skip 0 (the <blank> padding token) so it's readable
    if idx != 0: 
        decoded_text += idx_to_char.get(idx, "?")

print("\n--- First Item ---")
print(f"Raw Integer Tensor: {first_label.numpy()}")
print(f"Decoded Text: '{decoded_text}'")
print(f"Original Label Length: {first_length}")

# 6. Plot the image
fig, ax = plt.subplots(figsize=(10, 3))

# PyTorch images are (Channels, Height, Width)
# Matplotlib needs (Height, Width, Channels), so we use .permute()
image_to_show = first_image.permute(1, 2, 0).numpy()

# If it's grayscale (1 channel), squeeze it to 2D for matplotlib
if image_to_show.shape[2] == 1:
    image_to_show = image_to_show.squeeze()
    ax.imshow(image_to_show, cmap='gray')
else:
    ax.imshow(image_to_show)

ax.set_title(f"Label: {decoded_text}", fontsize=16)
ax.axis('off')
plt.show()
